# AuthAgent: Colab runner

Cost-aware agentic RAG for prior authorization with a self-trained verifier cascade.

**Runtime:** CPU is fine for everything except training the verifier. Switch to a **T4 GPU**
(Runtime → Change runtime type) before the training section, or run it in a separate session.

Order of work:
1. Setup (code, packages, Drive for persistence, LLM key)
2. Self-test and one smoke case
3. Train and calibrate the verifier (GPU, ~5 min)
4. Run the configurations and build the report
5. Demo

## 1. Setup
Get the code: set `REPO_URL` to your GitHub repo, or leave it empty and upload `authagent.zip`.

In [ ]:
import os
REPO_URL = ""   # e.g. "https://github.com/<you>/authagent.git"

if REPO_URL:
    if not os.path.exists("/content/authagent"):
        !git clone -q {REPO_URL} /content/authagent
    else:
        !git -C /content/authagent pull -q
else:
    if not os.path.exists("/content/authagent.zip"):
        from google.colab import files
        files.upload()   # choose authagent.zip
    !unzip -q -o /content/authagent.zip -d /content
%cd /content/authagent

In [ ]:
!pip install -q -r requirements.txt

Keep the LLM cache, results and the trained model on Google Drive, so a disconnect never
costs you a rerun and your teammate can reuse the same cache.

In [ ]:
import shutil
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/authagent_state"
    for d in ("cache", "results", "models"):
        os.makedirs(f"{BASE}/{d}", exist_ok=True)
    # Seed the Drive cache from the repo's committed cache the first time.
    if os.path.exists("cache/llm_cache.jsonl") and not os.path.exists(f"{BASE}/cache/llm_cache.jsonl"):
        shutil.copy("cache/llm_cache.jsonl", f"{BASE}/cache/llm_cache.jsonl")
    os.environ["AUTHAGENT_CACHE_DIR"] = f"{BASE}/cache"
    os.environ["AUTHAGENT_RESULTS_DIR"] = f"{BASE}/results"
    os.environ["AUTHAGENT_MODELS_DIR"] = f"{BASE}/models"
RESULTS = os.environ.get("AUTHAGENT_RESULTS_DIR", "results")
print("results ->", RESULTS)

### LLM configuration
Add your key in Colab's **Secrets** panel (key icon in the left sidebar) as `LLM_API_KEY`, and
allow this notebook to access it. Any OpenAI-compatible provider works. Model ids change over
time, so check your provider's current model list.

In [ ]:
from google.colab import userdata
os.environ["LLM_BASE_URL"] = "https://api.groq.com/openai/v1"   # or https://openrouter.ai/api/v1
os.environ["LLM_MODEL"] = "llama-3.3-70b-versatile"             # check the provider's model list
os.environ["LLM_API_KEY"] = userdata.get("LLM_API_KEY")
os.environ["LLM_MIN_INTERVAL_S"] = "2"   # spacing between calls for free-tier rate limits
os.environ["LLM_JSON_MODE"] = "1"

**Optional: no key, no limits.** Run an open model with Ollama inside Colab (use a GPU runtime;
slower than an API but unlimited). Skip this cell if you use an API key.

In [ ]:
USE_OLLAMA = False
if USE_OLLAMA:
    import subprocess, time
    !curl -fsSL https://ollama.com/install.sh | sh
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    !ollama pull qwen2.5:7b-instruct
    os.environ.update(LLM_BASE_URL="http://localhost:11434/v1", LLM_API_KEY="ollama",
                      LLM_MODEL="qwen2.5:7b-instruct", LLM_MIN_INTERVAL_S="0")

## 2. Self-test and a smoke case

In [ ]:
!python run.py selftest

In [ ]:
# One real case end to end: watch the agent search, call the rule engine, verify, decide.
!python run.py case P-001 --config agent

## 3. Train and calibrate the verifier (GPU)
Trains DeBERTa-v3-small on RA + GLP-1 pairs, fits a temperature and the cascade threshold on
validation, and reports on the **held-out PCSK9 policy**. `--compare-nli` also scores the
off-the-shelf NLI model on the same test set for your comparison slide.

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader || echo "No GPU: switch the runtime to T4 for this cell"

In [ ]:
# Smoke run first (1 epoch) to catch problems early; then the real run.
!python scripts/train_verifier.py --epochs 1 --out /tmp/verifier_smoke

In [ ]:
!python scripts/train_verifier.py --compare-nli

In [ ]:
import json
rep = json.load(open(f"{RESULTS}/verifier_report.json"))
for k in ("val_uncalibrated", "val_calibrated", "test_heldout_policy_uncalibrated", "test_heldout_policy_calibrated", "nli_offtheshelf_test"):
    if k in rep: print(f"{k:36s}", {m: round(v, 3) for m, v in rep[k].items() if m != "n"})
print("temperature", rep["temperature"], "tau", rep["tau"], "test@tau", rep["test_at_tau"])

## 4. Run the configurations
Start with 5 cases per configuration to check rate limits and costs, then run everything.
Runs resume automatically after a disconnect.

In [ ]:
!python run.py run --config all --limit 5

In [ ]:
!python run.py run --config all

In [ ]:
# Ablations: no rule engine; local verifier only (no LLM escalation)
!python run.py run --config agent_no_rules,agent_local_only

In [ ]:
!python run.py report
from IPython.display import Markdown, display
display(Markdown(open(f"{RESULTS}/report.md").read()))

In [ ]:
# Retrieval quality: BM25 vs dense vs hybrid (recall@k, MRR)
!python run.py retrieval

### Charts for the deck

In [ ]:
import json, matplotlib.pyplot as plt
r = json.load(open(f"{RESULTS}/report.json"))["configs"]
order = [c for c in ["single_prompt", "fixed_pipeline", "agent", "agent_cascade"] if c in r]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].bar(order, [100 * r[c]["decision_accuracy"] for c in order]); ax[0].set_title("Decision accuracy (%)")
ax[1].bar(order, [100 * (r[c]["false_approval_rate"] or 0) for c in order]); ax[1].set_title("False approvals (%)")
ax[2].bar(order, [r[c]["llm_calls_per_case"] for c in order]); ax[2].set_title("LLM calls per case")
for a in ax: a.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.savefig(f"{RESULTS}/headline.png", dpi=160); plt.show()

rep = json.load(open(f"{RESULTS}/verifier_report.json"))
rc = rep["test_risk_coverage"]
plt.figure(figsize=(5, 4))
plt.plot([x["coverage"] for x in rc], [100 * x["risk"] for x in rc], marker=".")
plt.xlabel("Coverage (share of checks the small model answers)"); plt.ylabel("Error on answered checks (%)")
plt.title("Verifier risk-coverage (held-out policy)"); plt.grid(alpha=.3)
plt.savefig(f"{RESULTS}/risk_coverage.png", dpi=160); plt.show()

## 5. Demo
Opens a Gradio app with a public link you can use in the interview (cached runs are instant).

In [ ]:
!python run.py demo --share